**一、指令模板**

这是我们智能体的“说明书”，它将作为 ```system_prompt``` 传递给 LLM，告诉 LLM 它应该扮演什么角色、拥有哪些工具、以及如何格式化它的思考和行动。

In [28]:
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行规划助手。你的任务是帮助用户规划完整的旅行，包括景点、交通、住宿等。你需要与用户交互，收集必要信息，然后使用可用工具一步步地获取数据，最终生成详细的行程计划。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。
- `recommend_accommodation(city, attractions, weather, preferences="")`: 查询指定城市的旅游者最多选择的居住地点（地铁站或景点附近）。
- `get_coordinate(place_name, city)`:查询景点和酒店的位置坐标。
- `get_travel_time_matrix(origins, destinations)`:查询两个地点之间行程耗时。
- `group_attractions_by_time(attractions, time_matrix, max_daily_travel=180)`:安排每日的行程安排，只包括每天去哪些景点。
- `plan_route(origin_coord, dest_coord, mode='transit', city='北京', fallback_to_driving=False)`: 获取两地之间的交通建议（如从酒店到景点）。
- `remember_preference(key: str, value: str)`: 记录用户的个性化偏好（如目的地、天数、兴趣、预算等），以便后续使用。
- `ask_user(question: str)`: 向用户提问以获取缺失的信息，例如目的地、旅行天数等。当你需要更多信息时使用此工具。

# 输出格式要求:
你的每次回复必须严格遵循以下格式，包含一对Thought和Action：

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action 的格式必须是以下之一：
1. 调用工具：function_name(arg_name=arg_value)
   - arg_value 可以是字符串（用双引号包裹）、数字、布尔值（True/False）、列表（用方括号）或字典（用花括号）。
   例如：get_attraction(city="北京", weather="晴天")
         group_attractions_by_time(attractions=["故宫", "天安门"], max_daily_travel=180)
2. 结束任务：Finish[最终答案]

# 重要提示:
- 你**必须**通过调用工具来获取所有信息，**绝对不允许**自己编造天气、景点、住宿等数据。
- 在调用任何工具之前，如果缺少必要信息（如目的地、天数），必须先使用 `ask_user` 询问用户。
- 你的每一步输出**必须**包含一对 `Thought:` 和 `Action:`，不要添加额外内容。且 `Action` 必须是工具调用或 `Finish[]`。
- 如果你试图不调用工具而直接输出行程，将导致严重错误。
- Action必须在同一行，不要换行
- get_travel_time_matrix(origins, destinations): 查询两个地点之间行程耗时。参数 origins 和 destinations 必须是 JSON 字符串，例如 origins="[[39.913951,116.410253],[39.917839,116.397029]]"。
- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[最终答案] 格式结束
- 在调用 recommend_accommodation 之前，请务必先通过 remember_preference 记录景点和天气信息（键名分别为 'attractions' 和 'weather'），或者确保调用时传入了 attractions 和 weather 参数。
- 最终答案应该是一个详细的行程计划，包括每日的景点安排和交通方式。
请开始吧！
"""

**二、工具**

1.查询天气

我们将通过高德地图天气 API 查询指定城市的实时天气，并返回格式化的天气信息或错误提示。这里也可以使用其他工具，例如免费的天气查询服务 wttr.in ，它能以 JSON 格式返回指定城市的天气数据。下面是实现该工具的代码：

In [29]:
import requests

def get_weather(city: str) -> str:
    """
    使用高德地图天气 API 查询实时天气。
    """
    api_key = os.getenv("GAODE_API_KEY")
    if not api_key:
        return "错误:未配置高德 API Key"
    
    url = "https://restapi.amap.com/v3/weather/weatherInfo"
    params = {
        "key": api_key,
        "city": city,
        "extensions": "base",
        "output": "json"
    }
    try:
        resp = requests.get(url, params=params, timeout=5)
        data = resp.json()
        if data["status"] == "1" and data.get("lives"):
            live = data["lives"][0]
            weather = live["weather"]
            temperature = live["temperature"]
            return f"{city}当前天气:{weather}，气温{temperature}摄氏度"
        else:
            return f"错误:获取天气失败 - {data.get('info', '未知错误')}"
    except Exception as e:
        return f"错误:查询天气时出现异常 - {e}"




2.搜索并推荐旅游景点

我们将定义一个新工具 search_attraction ，它会基于给定的城市和天气条件，利用 Tavily 搜索引擎获取并返回该城市在特定天气下的旅游景点推荐。

In [30]:
import os
from tavily import TavilyClient

def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)

    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"

    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答
        response = tavily.search(query=query, search_depth="basic",include_answer=True)
        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]

        # 如果没有综合性回答，则格式化原始结果
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")

        if not formatted_results:
            return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"
        

3.查询酒店

下面我们将基于城市、景点、天气和用户偏好，通过大语言模型生成住宿区域推荐，代码如下：

In [31]:
def recommend_accommodation(city, attractions=None, weather=None, preferences=""):
    # 如果参数未提供，尝试从用户偏好中获取
    if attractions is None:
        attractions = _user_preferences.get('attractions', '')
    if weather is None:
        weather = _user_preferences.get('weather', '')
    
    # 若仍然缺失关键信息，返回错误提示
    if not attractions or not weather:
        return "错误: 缺少景点或天气信息，无法推荐住宿。请先获取景点和天气数据。"
    
    prompt = f"""
你是一个经验丰富的旅游顾问。基于以下信息，为用户推荐住宿区域（只需推荐地铁站或者景点附近，不用具体酒店名称），并解释为什么大多数人会选择那里。
目的地：{city}
计划游览的景点：{attractions}
当地天气：{weather}
用户其他要求：{preferences}
请给出1-3个推荐的地铁站或区域，并简要说明理由。
"""
    # 使用已有的 LLM 客户端生成回答（假设全局有 llm 对象）
    response = llm.generate(prompt, system_prompt="你是一个旅游顾问，请简洁回答。")
    return response

4.日程安排


首先，我们利用使用高德地图地理编码 API 将指定的地点名称（如景点、酒店名）转换为地理坐标（纬度和经度），代码如下：

In [32]:
import requests
import time
def get_coordinate(place_name, city,api_key=None):
    """
    使用高德地图地理编码 API 查询地点坐标。
    优先使用传入的 api_key，若无效或未传入则尝试从环境变量获取。
    返回坐标元组 (纬度, 经度) 或错误字符串。
    """
    # 如果未传入 api_key 或传入的 key 无效（由后续调用判断），尝试从环境变量获取
    if api_key is None:
        api_key = os.getenv("GAODE_API_KEY")
        if api_key is None:
            return "错误：未配置高德 API Key，请在环境变量中设置 GAODE_API_KEY"
    # 注意：即使传入了 api_key，也可能无效，我们将在请求后统一处理错误

    url = "https://restapi.amap.com/v3/geocode/geo"
    params = {
        "key": api_key,
        "address": place_name,
        "city": city,
        "output": "json"
    }
    try:
        resp = requests.get(url, params=params, timeout=5)
        data = resp.json()
    except Exception as e:
        return f"错误: 请求高德地理编码 API 时发生异常 - {e}"

    # 检查 API 返回状态
    if data.get("status") != "1":
        info = data.get("info", "未知错误")
        if info == "INVALID_USER_KEY":
            # 如果传入的 key 无效且不是从环境变量来的，可以尝试使用环境变量重试一次
            if api_key != os.getenv("GAODE_API_KEY") and os.getenv("GAODE_API_KEY"):
                print("传入的 API Key 无效，尝试使用环境变量中的 Key 重试...")
                return get_coordinate(place_name, city, api_key=os.getenv("GAODE_API_KEY"))
            return f"错误: 高德地理编码失败 - {info}，请检查 API Key 是否有效并已开通服务"
        else:
            return f"错误: 高德地理编码失败 - {info}"

    geocodes = data.get("geocodes")
    if not geocodes:
        return f"错误: 未找到地点 '{place_name}' 的坐标信息"

    location = geocodes[0].get("location")  # 格式 "经度,纬度"
    if not location:
        return f"错误: 地点 '{place_name}' 坐标数据为空"

    lng, lat = map(float, location.split(','))
    return (lng, lat)  # 返回 (经度, 纬度) 元组



有了坐标，你需要知道任意两个景点之间的交通时间（或距离），方便我们对景点按天进行分类。下面我们使用高德地图距离测量 API，计算多个起点到多个终点之间的驾车耗时，并以二维矩阵形式返回。

In [33]:
def get_travel_time_matrix(origins, destinations, api_key=None):
    """
    查询多个起点到多个终点的驾车耗时矩阵。
    
    Args:
        origins (str): JSON 字符串，表示起点坐标列表，每个坐标为 [经度, 纬度] 的列表，例如 "[[116.411571,39.908069], [116.401304,39.905374]]"
        destinations (str): JSON 字符串，表示终点坐标列表，同样为经度,纬度格式
        api_key (str, optional): 高德 API Key，默认从环境变量获取
    
    Returns:
        str: 成功时返回耗时矩阵的 JSON 字符串（二维列表），失败时返回错误信息（以 "错误:" 开头）
    """
    import json
    import os
    import requests

    # 1. 解析 JSON 字符串为 Python 列表
    try:
        origins_list = json.loads(origins)
        destinations_list = json.loads(destinations)
    except json.JSONDecodeError as e:
        return f"错误: 解析 origins 或 destinations 失败 - {e}"
    
    # 2. 检查 API Key
    if api_key is None:
        api_key = os.getenv("GAODE_API_KEY")
        if api_key is None:
            return "错误：未配置高德 API Key"
    
    # 3. 构建高德距离矩阵 API 请求参数
    url = "https://restapi.amap.com/v3/distance"
    params = {
        "key": api_key,
        "type": "1",  # 1 表示驾车距离
        "origins": "|".join([f"{lng},{lat}" for lng, lat in origins_list]),  # 注意顺序：经度,纬度
        "destinations": "|".join([f"{lng},{lat}" for lng, lat in destinations_list]),
        "output": "json"
    }
    
    # 4. 发送请求
    try:
        resp = requests.get(url, params=params, timeout=5)
        data = resp.json()
    except Exception as e:
        return f"错误: 请求高德距离矩阵 API 时发生异常 - {e}"
    
    # 5. 处理响应
    if data.get("status") != "1":
        return f"错误: 高德距离矩阵请求失败 - {data.get('info', '未知错误')}"
    
    results = data.get("results")
    if not results:
        return "错误: 返回结果为空"
    
    # 6. 将结果组装为二维耗时矩阵（秒）
    matrix = []
    num_dests = len(destinations_list)
    for i in range(len(origins_list)):
        row = []
        for j in range(num_dests):
            item = results[i * num_dests + j]
            row.append(float(item["duration"]))  # 耗时（秒）
        matrix.append(row)
    
    # 返回矩阵的 JSON 字符串
    return json.dumps(matrix)

下面我们使用简单的贪心分组法来安排每日行程，选择一个起始景点（例如住宿最近的那个），找出与它耗时最短的景点，
如果总耗时不超过每天合理游览时间（例如 3-4 小时交通+游览），就加入当天，重复直到当天无法再增加，然后开始下一天。

In [34]:
def group_attractions_by_time(attractions, time_matrix, max_daily_travel=180):
    """
    根据景点间耗时矩阵，将景点分组到每天（贪心算法）。
    
    Args:
        attractions (str): JSON 字符串，表示景点名称列表，例如 '["中国国家博物馆","国家艺术博物馆"]'
        time_matrix (str): JSON 字符串，表示景点间耗时矩阵（秒），例如 '[[0,471],[471,0]]'
        max_daily_travel (int/str): 每天允许的最大交通耗时（分钟），默认 180 分钟。允许传入字符串，函数内部会自动转换。
    
    Returns:
        str: 成功时返回分组结果的 JSON 字符串（二维列表，每组为景点名称列表），失败时返回错误信息
    """
    import json

    # 1. 解析参数
    try:
        attractions_list = json.loads(attractions)
        time_matrix_list = json.loads(time_matrix)
    except json.JSONDecodeError as e:
        return f"错误: 解析 attractions 或 time_matrix 失败 - {e}"
    
    # 2. 处理 max_daily_travel（兼容字符串或数字）
    try:
        max_daily_travel = int(max_daily_travel)
    except (ValueError, TypeError):
        return f"错误: max_daily_travel 必须为整数，无法转换 '{max_daily_travel}'"
    
    # 3. 检查数据维度
    n = len(attractions_list)
    if not isinstance(time_matrix_list, list) or len(time_matrix_list) != n:
        return f"错误: time_matrix 维度不匹配，应为 {n}x{n} 的二维列表"
    for row in time_matrix_list:
        if not isinstance(row, list) or len(row) != n:
            return f"错误: time_matrix 维度不匹配，应为 {n}x{n} 的二维列表"
    
    # 4. 贪心分组算法
    visited = [False] * n
    groups = []
    
    max_daily_travel_sec = max_daily_travel * 60  # 转为秒
    
    for i in range(n):
        if visited[i]:
            continue
        # 开始新的一天
        day = [attractions_list[i]]
        visited[i] = True
        total_travel = 0
        current = i
        while True:
            # 找到与 current 耗时最短的未访问景点
            best_j = -1
            best_time = float('inf')
            for j in range(n):
                if not visited[j] and time_matrix_list[current][j] < best_time:
                    best_time = time_matrix_list[current][j]
                    best_j = j
            if best_j == -1:
                break
            # 如果加上这个耗时还允许，就加入当天
            if total_travel + best_time <= max_daily_travel_sec:
                day.append(attractions_list[best_j])
                visited[best_j] = True
                total_travel += best_time
                current = best_j
            else:
                break
        groups.append(day)
    
    # 返回分组结果的 JSON 字符串
    return json.dumps(groups, ensure_ascii=False)

下面的函数对路径进行规划，使用高德地图，选择合适的交通方式，包含耗时（分钟）、距离（公里）、路线。由于我未开通公交权限，无法使用公交规划，增加了一个辅助函数 ``plan_route_with_driving_fallback`` ，当 ```mode='transit'``` 且请求失败时，若启用了回退，则自动调用驾车模式重新规划，并在返回结果中添加 ``fallback_note`` 字段说明情况。

In [35]:
import requests

def plan_route(origin_coord, dest_coord, mode='transit', city='北京', fallback_to_driving=False,api_key=None):
    """高德路径规划API调用函数（支持公交不可用时自动回退到驾车）"""
    if api_key is None:
        api_key = os.getenv("GAODE_API_KEY")
        if api_key is None:
            return "错误：未配置高德 API Key"
    origin = f"{origin_coord[0]},{origin_coord[1]}"
    destination = f"{dest_coord[0]},{dest_coord[1]}"
    
    fallback_used = False
    fallback_note = None

    if mode == 'transit' and fallback_to_driving:
        transit_result = _call_api(origin, destination, mode='transit', city=city, api_key=api_key)
        if transit_result is not None:
            return transit_result
        else:
            fallback_used = True
            fallback_note = "公交规划不可用（可能未开通权限），已自动切换为驾车模式。"
            print(fallback_note)
            mode = 'driving'
    
    result = _call_api(origin, destination, mode=mode, city=city, api_key=api_key)
    if result and fallback_used:
        result['fallback_note'] = fallback_note
    return result

def _call_api(origin, destination, mode, city=None,api_key=None):
    """实际调用高德API的内部函数，返回结果字典或None"""
    if api_key is None:
        api_key = os.getenv("GAODE_API_KEY")
        if api_key is None:
            return "错误：未配置高德 API Key"
    
    if mode == 'transit':
        url = "https://restapi.amap.com/v3/direction/integrated"
        params = {
            'key': api_key,
            'origin': origin,
            'destination': destination,
            'city': city,
            'strategy': 0,
            'nightflag': 0,
            'output': 'json'
        }
    elif mode == 'driving':
        url = "https://restapi.amap.com/v3/direction/driving"
        params = {
            'key': api_key,
            'origin': origin,
            'destination': destination,
            'strategy': 0,
            'extensions': 'all',
            'output': 'json'
        }
    elif mode == 'walking':
        url = "https://restapi.amap.com/v3/direction/walking"
        params = {
            'key': api_key,
            'origin': origin,
            'destination': destination,
            'output': 'json'
        }
    elif mode == 'cycling':
        url = "https://restapi.amap.com/v3/direction/cycling"
        params = {
            'key': api_key,
            'origin': origin,
            'destination': destination,
            'output': 'json'
        }
    else:
        return None

    try:
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        if data.get('status') != '1':
            print(f"API错误 ({mode}): {data.get('info', '未知错误')}")
            return None
        route = data['route']
        # ... 后续解析代码与原单元格相同（省略，保持原样）
    except Exception as e:
        print(f"请求异常 ({mode}): {e}")
        return None

In [36]:
# 用于存储用户偏好的全局字典
_user_preferences = {}

def remember_preference(key: str, value: str) -> str:
    """
    记录用户的个性化偏好（如目的地、天数、兴趣、预算等），以便后续使用。

    Args:
        key (str): 偏好项的名称，例如 'destination', 'days', 'interests', 'budget'。
        value (str): 偏好的具体值，由用户提供或从对话中提取。
    """
    _user_preferences[key] = value
    # 可选：打印确认信息（便于调试）
    #print(f"已记录偏好：{key} = {value})
    return f"已记录偏好：{key} = {value}"

下面定义一个简单的 ask_user 函数，用于向用户提问并获取输入。该函数适用于命令行环境，通过 ```input()``` 获取用户回答，并返回清理后的字符串。

In [37]:
def ask_user(question: str) -> str:
    """
    向用户提问以获取缺失的信息，例如目的地、旅行天数等。
    当你需要更多信息时使用此工具。

    Args:
        question (str): 要向用户提出的问题。

    Returns:
        str: 用户的回答（已去除首尾空白）。
    """
    print(question)
    answer = input().strip()
    return answer

In [38]:
# 将所有工具函数放入一个字典，方便后续调用
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
    "recommend_accommodation":recommend_accommodation,
    "get_coordinate":get_coordinate,
    "get_travel_time_matrix":get_travel_time_matrix,
    "group_attractions_by_time":group_attractions_by_time,
    "plan_route":plan_route,
    "ask_user": ask_user,
    "remember_preference": remember_preference
}

In [39]:
from openai import OpenAI

class OpenAICompatibleClient:
    """
    一个用于调用任何兼容OpenAI接口的LLM服务的客户端。
    """
    def __init__(self, model: str, api_key: str, base_url: str):
        self.model = model
        self.client = OpenAI(api_key=api_key, base_url=base_url)

    def generate(self, prompt: str, system_prompt: str) -> str:
        """调用LLM API来生成回应。"""
        print("正在调用大语言模型...")
        try:
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt}
            ]
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                stream=False
            )
            answer = response.choices[0].message.content
            print("大语言模型响应成功。")
            return answer
        except Exception as e:
            print(f"调用LLM API时发生错误: {e}")
            return "错误:调用语言模型服务时出错。"

In [40]:
import re
import os
import json

# --- 1. 配置LLM客户端 ---
# 请根据您使用的服务，将这里替换成对应的凭证和地址
API_KEY = "sk-CtJ1fRV7T4R4HigI85F6D6A145A6431bBfCb3dC511D8322d"
BASE_URL = "https://aihubmix.com/v1"
MODEL_ID = "coding-glm-4.7-free"
TAVILY_API_KEY="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"
os.environ['TAVILY_API_KEY'] = "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"
os.environ["GAODE_API_KEY"] = "4649eb59d515dd82e5611e6ee953f2ec"

llm = OpenAICompatibleClient(
    model=MODEL_ID,
    api_key=API_KEY,
    base_url=BASE_URL
)

# --- 2. 初始化 ---
preference = []
print("\n请输入您的旅行需求（例如：我想去北京玩3天，喜欢历史文化）：")
user_prompt = input().strip()
prompt_history = [f"用户说：{user_prompt}"]
# --- 3. 运行主循环 ---
for i in range(30): # 设置最大循环次数
    print(f"--- 循环 {i+1} ---\n")
    # 3.1. 构建Prompt
    full_prompt = "\n".join(prompt_history)

    # 3.2. 调用LLM进行思考
    llm_output = llm.generate(full_prompt, system_prompt=AGENT_SYSTEM_PROMPT)
    # 模型可能会输出多余的Thought-Action，需要截断
    match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', llm_output, re.DOTALL)
    if match:
        truncated = match.group(1).strip()
        if truncated != llm_output.strip():
            llm_output = truncated
            print("已截断多余的 Thought-Action 对")
    print(f"模型输出:\n{llm_output}\n")
    prompt_history.append(llm_output)
    # 3.3. 解析并执行行动
        # 3.3. 解析并执行行动
    action_match = re.search(r"Action: (.*)", llm_output, re.DOTALL)
    if not action_match:
        observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ...Action: ...' 的格式。"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue
    action_str = action_match.group(1).strip()

    if action_str.startswith("Finish"):
        final_match = re.search(r"Finish\[(.*)\]", action_str, re.DOTALL)
        if final_match:
            final_answer = final_match.group(1)
            print(f"任务完成，最终答案: {final_answer}")
            break
        else:
            observation = "错误: Finish 格式不正确，应为 Finish[最终答案]"
            observation_str = f"Observation: {observation}"
            print(f"{observation_str}\n" + "="*40)
            prompt_history.append(observation_str)
            continue

    # 提取工具名称
    tool_name_match = re.search(r"(\w+)\(", action_str)
    if not tool_name_match:
        observation = f"错误: 无法从 Action 中解析出工具名称: {action_str}"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue
    tool_name = tool_name_match.group(1)
    print(f"解析到工具名称: {tool_name}")

    # 提取参数部分
    args_match = re.search(r"\((.*)\)", action_str, re.DOTALL)
    if not args_match:
        observation = f"错误: Action 缺少参数括号或参数格式错误: {action_str}"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue
    args_str = args_match.group(1).strip()
    print(f"参数原始字符串: {args_str}")

    # 尝试解析命名参数：key="value"
    kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))
    if kwargs:
        print(f"解析到命名参数: {kwargs}")
    else:
        # 尝试解析位置参数（双引号字符串）
        quoted_strings = re.findall(r'"([^"]*)"', args_str)
        print(f"解析到位置参数列表: {quoted_strings}")
        if not quoted_strings:
            observation = f"错误: 无法解析工具 '{tool_name}' 的参数，未找到任何引号内的内容。"
            observation_str = f"Observation: {observation}"
            print(f"{observation_str}\n" + "="*40)
            prompt_history.append(observation_str)
            continue

        # 根据工具名称将位置参数映射到参数名
        if tool_name == "ask_user":
            if len(quoted_strings) >= 1:
                kwargs = {"question": quoted_strings[0]}
        elif tool_name == "get_weather":
            if len(quoted_strings) >= 1:
                kwargs = {"city": quoted_strings[0]}
        elif tool_name == "get_attraction":
            if len(quoted_strings) >= 2:
                kwargs = {"city": quoted_strings[0], "weather": quoted_strings[1]}
        elif tool_name == "remember_preference":
            if len(quoted_strings) >= 2:
                kwargs = {"key": quoted_strings[0], "value": quoted_strings[1]}
        elif tool_name == "recommend_accommodation":
            # 按顺序：city, attractions, weather, preferences
            if len(quoted_strings) >= 1:
                kwargs = {"city": quoted_strings[0]}
            if len(quoted_strings) >= 2:
                kwargs["attractions"] = quoted_strings[1]
            if len(quoted_strings) >= 3:
                kwargs["weather"] = quoted_strings[2]
            if len(quoted_strings) >= 4:
                kwargs["preferences"] = quoted_strings[3]
        elif tool_name == "get_coordinate":
            if len(quoted_strings) >= 2:
                kwargs = {"place_name": quoted_strings[0], "city": quoted_strings[1]}
            # api_key 可选，可省略
        elif tool_name == "get_travel_time_matrix":
            observation = f"错误: 工具 '{tool_name}' 不支持位置参数，请使用命名参数。"
            observation_str = f"Observation: {observation}"
            print(f"{observation_str}\n" + "="*40)
            prompt_history.append(observation_str)
            continue
        elif tool_name == "group_attractions_by_time":
            observation = f"错误: 工具 '{tool_name}' 不支持位置参数，请使用命名参数。"
            observation_str = f"Observation: {observation}"
            print(f"{observation_str}\n" + "="*40)
            prompt_history.append(observation_str)
            continue
        elif tool_name == "plan_route":
            observation = f"错误: 工具 '{tool_name}' 不支持位置参数，请使用命名参数。"
            observation_str = f"Observation: {observation}"
            print(f"{observation_str}\n" + "="*40)
            prompt_history.append(observation_str)
            continue
        else:
            observation = f"错误: 无法解析 Action 的参数，工具 '{tool_name}' 需要命名参数或未定义。"
            observation_str = f"Observation: {observation}"
            print(f"{observation_str}\n" + "="*40)
            prompt_history.append(observation_str)
            continue

    # 检查是否成功构造了 kwargs
    if not kwargs:
        observation = f"错误: 无法解析工具 '{tool_name}' 的参数，可能参数数量不足。"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue

    # 调用工具
    if tool_name in available_tools:
        try:
            print(f"正在调用工具 {tool_name}，参数: {kwargs}")
            observation = available_tools[tool_name](**kwargs)
            print(f"工具返回: {observation}")
        except TypeError as e:
            observation = f"错误: 调用工具 '{tool_name}' 时参数不匹配: {e}。提供的参数: {kwargs}"
        except Exception as e:
            observation = f"错误: 调用工具 '{tool_name}' 时发生异常: {e}"
    else:
        observation = f"错误: 未定义的工具 '{tool_name}'"

    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)


请输入您的旅行需求（例如：我想去北京玩3天，喜欢历史文化）：


 我想去北京玩3天，喜欢历史文化


--- 循环 1 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 用户提供了目的地（北京）、天数（3天）和兴趣偏好（历史文化）。但缺少旅行日期，我需要知道日期来查询天气，因为天气会影响景点推荐和住宿建议。我应该先询问用户的旅行日期。

Action: ask_user(question="请问您计划什么时候去北京旅行呢？我需要查询天气来为您推荐合适的景点和住宿。")

解析到工具名称: ask_user
参数原始字符串: question="请问您计划什么时候去北京旅行呢？我需要查询天气来为您推荐合适的景点和住宿。"
解析到命名参数: {'question': '请问您计划什么时候去北京旅行呢？我需要查询天气来为您推荐合适的景点和住宿。'}
正在调用工具 ask_user，参数: {'question': '请问您计划什么时候去北京旅行呢？我需要查询天气来为您推荐合适的景点和住宿。'}
请问您计划什么时候去北京旅行呢？我需要查询天气来为您推荐合适的景点和住宿。


 3.1-3.3


工具返回: 3.1-3.3
Observation: 3.1-3.3
--- 循环 2 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 用户提供了旅行日期（3月1日-3月3日）。现在我有了足够的基本信息。让我先记录用户的偏好，然后查询北京的天气情况，因为天气会影响景点推荐和住宿建议。

Action: remember_preference(key="destination", value="北京")

解析到工具名称: remember_preference
参数原始字符串: key="destination", value="北京"
解析到命名参数: {'key': 'destination', 'value': '北京'}
正在调用工具 remember_preference，参数: {'key': 'destination', 'value': '北京'}
工具返回: 已记录偏好：destination = 北京
Observation: 已记录偏好：destination = 北京
--- 循环 3 ---

正在调用大语言模型...
大语言模型响应成功。
已截断多余的 Thought-Action 对
模型输出:
Thought: 已记录目的地。现在我需要记录用户的旅行天数、兴趣偏好，然后查询北京的天气情况。

Action: remember_preference(key="days", value="3")

解析到工具名称: remember_preference
参数原始字符串: key="days", value="3"
解析到命名参数: {'key': 'days', 'value': '3'}
正在调用工具 remember_preference，参数: {'key': 'days', 'value': '3'}
工具返回: 已记录偏好：days = 3
Observation: 已记录偏好：days = 3
--- 循环 4 ---

正在调用大语言模型...
调用LLM API时发生错误: Error code: 429 - {'error': {'message': 'Sorry, you have reached the limit of the free model quota. Pleas